In [ ]:
# ============================================================
#  CELL 1 — Environment setup + all shared code
#  Run this cell first; every subsequent cell depends on it.
# ============================================================

import os
os.environ['MUJOCO_GL']         = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
os.environ['LD_PRELOAD']        = ''

import subprocess, sys

# IN_KAGGLE = os.path.exists('/kaggle/working')
# if IN_KAGGLE:
#     subprocess.run(['apt-get', 'install', '-y', '-q',
#                     'libgl1-mesa-glx', 'libosmesa6', 'libosmesa6-dev',
#                     'libglfw3'], capture_output=True)
#     print('✓ System GL libraries ready')

IN_KAGGLE = os.path.exists('/kaggle/working')
IN_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_KAGGLE or IN_COLAB:
    subprocess.run(['apt-get', 'install', '-y', '-q',
                    'libgl1-mesa-glx', 'libosmesa6', 'libosmesa6-dev',
                    'libglfw3', 'patchelf'], capture_output=True)
    print(f'✓ System GL libraries ready  ({"Colab" if IN_COLAB else "Kaggle"})')

def pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                   capture_output=True)

# for pkg in ['gymnasium', 'gymnasium[mujoco]', 'torch',
#             'numpy', 'matplotlib', 'pillow', 'imageio']:
#     pip(pkg)
for pkg in ['gymnasium', 'gymnasium[mujoco]', 'torch',
            'numpy', 'matplotlib', 'pillow', 'imageio']:
    pip(pkg)

# Colab-specific: ensure mujoco binary itself is present
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mujoco'],
                   capture_output=True)
print('✓ Python dependencies ready')

# ── Imports ──────────────────────────────────────────────────
import time, random, io, warnings, math
warnings.filterwarnings('ignore')

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import deque

import gymnasium as gym
from IPython.display import display, clear_output
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ══════════════════════════════════════════════════════════════
#  SDAC HYPERPARAMETERS
#  From Table 1 of Ma et al. 2025 (paper defaults for MuJoCo)
# ══════════════════════════════════════════════════════════════
SEED             = 42
TOTAL_STEPS      = 1_000_000   # 1M per variant (budget per task)
HIDDEN_SIZE      = 256         # score network width (paper default)
BATCH_SIZE       = 4096
REPLAY_SIZE      = 50_000
START_STEPS      = 10_000      # random warm-up
# UPDATES_PER_STEP = 1
UPDATES_PER_STEP = 50
GAMMA            = 0.99        # [ASSUMPTION: geometric discount]
TAU              = 0.005       # soft target update
LR_CRITIC        = 3e-4
LR_ACTOR         = 3e-4
LR_ALPHA         = 3e-4
ALPHA_INIT       = 0.2         # SAC-style temperature
TARGET_UPDATE_INTERVAL = 2     # paper uses 2 for SDAC

# DDPM / diffusion parameters
# Paper uses T=5 denoising steps for efficiency (Algorithm 1)
N_DIFFUSION_STEPS = 5          # [ASSUMPTION: 5-step DDPM sufficient for MuJoCo]
BETA_MIN          = 1e-4       # linear noise schedule
BETA_MAX          = 0.02
# N_SAMPLE_ACTIONS  = 16         # K: actions sampled for logsumexp (paper §4.2)
N_SAMPLE_ACTIONS  = 64

PLOT_EVERY   = 1_000
RENDER_EVERY = 1_000

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ══════════════════════════════════════════════════════════════
#  DDPM NOISE SCHEDULE
#  Standard linear beta schedule from Ho et al. 2020,
#  used verbatim in SDAC (paper §3, footnote 2).
# ══════════════════════════════════════════════════════════════

def make_ddpm_schedule(T, beta_min, beta_max, device):
    """
    Returns precomputed DDPM schedule tensors.
    q(a_t | a_0) = N(sqrt(ᾱ_t) a_0, (1-ᾱ_t) I)
    [ASSUMPTION: Gaussian forward process — tractable q(a_t|a_0)]
    """
    betas   = torch.linspace(beta_min, beta_max, T, device=device)  # β_1..β_T
    alphas  = 1.0 - betas                                            # α_t
    alpha_bar = torch.cumprod(alphas, dim=0)                        # ᾱ_t
    sqrt_ab   = alpha_bar.sqrt()                                    # √ᾱ_t
    sqrt_1mab = (1.0 - alpha_bar).sqrt()                            # √(1-ᾱ_t)
    return dict(betas=betas, alphas=alphas, alpha_bar=alpha_bar,
                sqrt_ab=sqrt_ab, sqrt_1mab=sqrt_1mab)


# ══════════════════════════════════════════════════════════════
#  SCORE NETWORK  (MLP variant — V0 baseline)
#
#  Architecture from paper §4 / DACER codebase:
#  Input:  (state, noisy_action_t, diffusion_timestep t)
#  Output: score estimate s_θ(a_t, s, t) ≈ ∇_{a_t} log π̃_t(a_t|s)
#
#  Time embedding: sinusoidal (same as DDPM Ho et al. 2020)
#  [ASSUMPTION: Markov — score conditioned on (s, a_t, t) only]
# ══════════════════════════════════════════════════════════════

class SinusoidalTimeEmbedding(nn.Module):
    """Sinusoidal timestep embedding from Ho et al. 2020."""
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        # t: (B,) integer or float in [0, T]
        half = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=t.device) / (half - 1)
        ).float()
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)  # (B, half)
        emb  = torch.cat([args.sin(), args.cos()], dim=-1)  # (B, dim)
        return emb


class ScoreNetworkMLP(nn.Module):
    """
    MLP score network s_θ(a_t, s, t).
    Used by V0 (baseline) and V2 (PER).

    [ASSUMPTION: Markov — no history, only current (s, a_t, t)]
    [ASSUMPTION: diagonal score — output same dim as action]

    Port note [PORT-1]: paper uses FiLM-style time conditioning in
    the JAX implementation; we use simple concatenation of sinusoidal
    embedding which is standard and equivalent in expressiveness.
    """
    def __init__(self, obs_dim, act_dim, hidden_dim=256, time_dim=32):
        super().__init__()
        self.time_emb = SinusoidalTimeEmbedding(time_dim)
        in_dim = obs_dim + act_dim + time_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim), nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim), nn.Mish(),
            nn.Linear(hidden_dim, act_dim),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    def forward(self, obs, noisy_action, t):
        # obs: (B, obs_dim), noisy_action: (B, act_dim), t: (B,)
        te  = self.time_emb(t)                    # (B, time_dim)
        x   = torch.cat([obs, noisy_action, te], dim=-1)
        return self.net(x)                        # (B, act_dim)


# ══════════════════════════════════════════════════════════════
#  LSTM SCORE NETWORK  (V1 — relaxes Markov assumption)
#
#  [RELAXED: Markov] Hidden state h_t carries history τ_{0:t}.
#  Same RSSM loss — only the score network architecture changes.
#  [PORT-2]: Standard LSTM wrapping of the score backbone;
#  not explicitly in paper but is the natural non-Markov extension.
# ══════════════════════════════════════════════════════════════

class ScoreNetworkLSTM(nn.Module):
    """
    [RELAXED: Markov] LSTM score network.
    Obs is encoded through LSTM before score head.
    """
    def __init__(self, obs_dim, act_dim, hidden_dim=256, time_dim=32):
        super().__init__()
        self.time_emb  = SinusoidalTimeEmbedding(time_dim)
        self.lstm      = nn.LSTM(obs_dim, hidden_dim, batch_first=True)
        in_dim = hidden_dim + act_dim + time_dim
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.Mish(),
            nn.Linear(hidden_dim, hidden_dim), nn.Mish(),
            nn.Linear(hidden_dim, act_dim),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=1.0)
                nn.init.zeros_(m.bias)

    def forward(self, obs, noisy_action, t, hidden=None):
        te = self.time_emb(t)
        h, hidden_out = self.lstm(obs.unsqueeze(1), hidden)  # (B,1,hidden)
        h = h.squeeze(1)                                     # (B, hidden)
        x = torch.cat([h, noisy_action, te], dim=-1)
        return self.head(x), hidden_out


# ══════════════════════════════════════════════════════════════
#  TWIN Q-NETWORK  (identical to SAC / pranz24)
#  SDAC reuses SAC-style twin critics unchanged (paper §4).
#  [ASSUMPTION: Markov — Q(s,a) sufficient]
#  [ASSUMPTION: scalar reward — single output per head]
# ══════════════════════════════════════════════════════════════

class QNetwork(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=256):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(obs_dim + act_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),         nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        self.q2 = nn.Sequential(
            nn.Linear(obs_dim + act_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),         nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, obs, action):
        x  = torch.cat([obs, action], dim=-1)
        return self.q1(x), self.q2(x)


# ══════════════════════════════════════════════════════════════
#  REPLAY MEMORY
# ══════════════════════════════════════════════════════════════

class ReplayMemory:
    """
    Uniform replay buffer.
    [ASSUMPTION: i.i.d. sampling — all transitions equally likely]
    """
    def __init__(self, capacity, seed):
        random.seed(seed)
        self.capacity = capacity
        self.buf      = []
        self.pos      = 0

    def push(self, s, a, r, ns, done):
        if len(self.buf) < self.capacity:
            self.buf.append(None)
        self.buf[self.pos] = (s, a, r, ns, done)
        self.pos = (self.pos + 1) % self.capacity

    # def sample(self, n):
    #     batch = random.sample(self.buf, n)
    #     s, a, r, ns, d = map(np.stack, zip(*batch))
    #     return s, a, r, ns, d
    def sample(self, n):
        indices = np.random.randint(0, len(self.buf), size=n)
        batch   = [self.buf[i] for i in indices]
        s, a, r, ns, d = map(lambda x: np.array(x, dtype=np.float32),
                            zip(*batch))
        return s, a, r, ns, d

    def __len__(self):
        return len(self.buf)


class PrioritizedReplayMemory:
    """
    Prioritized Experience Replay (Schaul et al. 2016).
    [RELAXED: i.i.d.] Transitions weighted by |TD error|.
    Uses sum-tree for O(log N) sampling.
    """
    def __init__(self, capacity, seed, alpha=0.6, beta_start=0.4):
        random.seed(seed)
        self.capacity   = capacity
        self.alpha      = alpha
        self.beta_start = beta_start
        self.eps        = 1e-6
        self.step       = 0
        # sum-tree
        self.tree_size  = 1
        while self.tree_size < capacity:
            self.tree_size *= 2
        self.tree    = np.zeros(2 * self.tree_size)
        self.data    = [None] * self.tree_size
        self.n_items = 0
        self.write   = 0
        self.max_p   = 1.0

    def _update(self, idx, p):
        pos = idx + self.tree_size
        self.tree[pos] = p
        pos //= 2
        while pos >= 1:
            self.tree[pos] = self.tree[2*pos] + self.tree[2*pos+1]
            pos //= 2

    def _sample_one(self, v):
        idx = 1
        while idx < self.tree_size:
            l = 2 * idx
            idx = l if v <= self.tree[l] else (idx := l+1, v := v - self.tree[l])[0]
        return idx - self.tree_size

    def push(self, s, a, r, ns, done):
        self.data[self.write] = (s, a, r, ns, done)
        self._update(self.write, self.max_p ** self.alpha)
        self.write   = (self.write + 1) % self.tree_size
        self.n_items = min(self.n_items + 1, self.tree_size)

    def sample(self, n):
        self.step += 1
        beta  = min(1.0, self.beta_start + (1-self.beta_start)*self.step/TOTAL_STEPS)
        total = self.tree[1]
        seg   = total / n
        idxs, priorities, batch = [], [], []
        for i in range(n):
            v   = random.uniform(seg*i, seg*(i+1))
            idx = self._sample_one(v)
            idxs.append(idx)
            priorities.append(self.tree[idx + self.tree_size])
            batch.append(self.data[idx])
        probs   = np.array(priorities) / total
        weights = (self.n_items * probs) ** (-beta)
        weights /= weights.max()
        weights  = torch.FloatTensor(weights).to(DEVICE)
        s, a, r, ns, d = map(np.stack, zip(*batch))
        return s, a, r, ns, d, idxs, weights

    def update_priorities(self, idxs, errors):
        for i, e in zip(idxs, errors):
            p = (abs(e) + self.eps) ** self.alpha
            self._update(i, p)
            self.max_p = max(self.max_p, p)

    def __len__(self):
        return self.n_items


# ══════════════════════════════════════════════════════════════
#  SDAC AGENT
#
#  Implements Algorithm 1 from Ma et al. 2025:
#
#  Critic update (standard SAC twin-Q, eq. 10-11 of paper):
#    y = r + γ (min Q̂(s',ã') − α log π(ã'|s'))
#    L_Q = E[(Q_i(s,a) − y)²]  for i=1,2
#
#  Actor update via RSSM (eq. 14-16 of paper):
#    Sample ã_0 ~ replay policy (current policy samples)
#    Compute noisy ã_t = √ᾱ_t ã_0 + √(1−ᾱ_t) ε,  ε~N(0,I)
#    Reweight by w(ã_0) = softmax_K( Q(s,ã_0)/α )   [logsumexp trick]
#    L_SDAC = E_{t,ã_t,ã_0}[ w(ã_0) · ||s_θ(ã_t,s,t) − (ã_t − √ᾱ_t ã_0)/√(1−ᾱ_t)||² ]
#    Note: the target is the negative of the conditional score:
#      ∇_{ã_t} log q(ã_t|ã_0) = −(ã_t − √ᾱ_t ã_0) / (1−ᾱ_t)
#    The loss minimises ||s_θ − (−noise/√(1−ᾱ_t))||² reweighted by w.
#    [PORT-3]: The paper shows this in score parameterisation;
#    we use the equivalent ε-prediction parameterisation (DDPM Ho 2020)
#    which is numerically more stable.
#
#  Temperature update (standard SAC auto-α, eq. 12 of paper):
#    L_α = −α (log π(ã|s) + H_target)
#
#  Action sampling (inference, Algorithm 1 line 4):
#    Run DDPM reverse process: ã_T~N(0,I), denoise T steps using s_θ
# ══════════════════════════════════════════════════════════════

def soft_update(tgt, src, tau):
    for tp, sp in zip(tgt.parameters(), src.parameters()):
        tp.data.copy_(tp.data * (1-tau) + sp.data * tau)

def hard_update(tgt, src):
    for tp, sp in zip(tgt.parameters(), src.parameters()):
        tp.data.copy_(sp.data)


class SDACAgent:
    """
    V0 — Baseline SDAC.
    All assumptions from Ma et al. 2025 intact.
    """

    def __init__(self, obs_dim, act_dim, action_space, schedule):
        self.obs_dim    = obs_dim
        self.act_dim    = act_dim
        self.schedule   = schedule
        self.T          = N_DIFFUSION_STEPS
        self.alpha      = ALPHA_INIT
        self.updates    = 0
        self.target_entropy = -float(act_dim)  # [ASSUMPTION: fixed H=-|A|]

        # Action rescaling (same as pranz24 GaussianPolicy)
        self.act_scale = torch.FloatTensor(
            (action_space.high - action_space.low) / 2.).to(DEVICE)
        self.act_bias  = torch.FloatTensor(
            (action_space.high + action_space.low) / 2.).to(DEVICE)

        # Score network (diffusion policy)
        self.score_net  = self._make_score_net(obs_dim, act_dim).to(DEVICE)
        self.score_opt  = Adam(self.score_net.parameters(), lr=LR_ACTOR)

        # Twin Q-critics + slow target
        self.critic     = QNetwork(obs_dim, act_dim, HIDDEN_SIZE).to(DEVICE)
        self.critic_tgt = QNetwork(obs_dim, act_dim, HIDDEN_SIZE).to(DEVICE)
        hard_update(self.critic_tgt, self.critic)
        self.critic_opt = Adam(self.critic.parameters(), lr=LR_CRITIC)

        # Auto-temperature
        self.log_alpha  = torch.zeros(1, requires_grad=True, device=DEVICE)
        self.alpha_opt  = Adam([self.log_alpha], lr=LR_ALPHA)

        self.score_net  = torch.compile(self.score_net,  mode='reduce-overhead')
        self.critic     = torch.compile(self.critic,      mode='reduce-overhead')
        self.critic_tgt = torch.compile(self.critic_tgt,  mode='reduce-overhead')

    def _make_score_net(self, obs_dim, act_dim):
        """Override in subclasses to swap architecture."""
        return ScoreNetworkMLP(obs_dim, act_dim, HIDDEN_SIZE)

    # ── DDPM reverse sampling (inference) ────────────────────
    # @torch.no_grad()
    # def _ddpm_sample(self, obs, hidden=None):
    #     """
    #     Algorithm 1, line 4: run reverse diffusion to get action.
    #     Starts from ã_T ~ N(0,I), denoises T steps.
    #     [ASSUMPTION: DDPM reverse process with learned score]
    #     """
    #     B  = obs.shape[0]
    #     at = torch.randn(B, self.act_dim, device=DEVICE)  # ã_T ~ N(0,I)

    #     for t_idx in reversed(range(self.T)):
    #         t_tensor = torch.full((B,), t_idx, device=DEVICE)
    #         score, hidden = self._score_forward(obs, at, t_tensor, hidden)

    #         # DDPM reverse step (Ho et al. 2020 eq. 11)
    #         beta_t     = self.schedule['betas'][t_idx]
    #         alpha_t    = self.schedule['alphas'][t_idx]
    #         alpha_bar_t= self.schedule['alpha_bar'][t_idx]
    #         sqrt_1mab  = self.schedule['sqrt_1mab'][t_idx]

    #         # ε-prediction: score network predicts noise ε
    #         # ã_{t-1} = (1/√α_t)(ã_t − β_t/√(1-ᾱ_t) · s_θ) + σ_t z
    #         coef    = beta_t / sqrt_1mab
    #         mean    = (1.0 / alpha_t.sqrt()) * (at - coef * score)

    #         if t_idx > 0:
    #             noise = torch.randn_like(at)
    #             at    = mean + beta_t.sqrt() * noise
    #         else:
    #             at = mean   # no noise at final step

    #     # Rescale from [-1,1] to action space
    #     at = torch.tanh(at) * self.act_scale + self.act_bias
    #     return at, hidden

    @torch.no_grad()
    def _ddpm_sample(self, obs, hidden=None):
        B  = obs.shape[0]
        at = torch.randn(B, self.act_dim, device=DEVICE)

        # Unroll the T=5 steps explicitly — no Python loop overhead
        # Each step is a single fused GPU operation
        for t_idx in reversed(range(self.T)):
            t_tensor   = torch.full((B,), t_idx, device=DEVICE, dtype=torch.long)
            score, hidden = self._score_forward(obs, at, t_tensor, hidden)

            beta_t    = self.schedule['betas'][t_idx]
            alpha_t   = self.schedule['alphas'][t_idx]
            sqrt_1mab = self.schedule['sqrt_1mab'][t_idx]
            coef      = beta_t / sqrt_1mab
            mean      = (1.0 / alpha_t.sqrt()) * (at - coef * score)
            at = mean + (beta_t.sqrt() * torch.randn_like(at) if t_idx > 0
                        else torch.zeros_like(at))

        return torch.tanh(at) * self.act_scale + self.act_bias, hidden

    def _score_forward(self, obs, noisy_action, t, hidden=None):
        """Forward pass of score network. Returns (score, hidden)."""
        score = self.score_net(obs, noisy_action, t)
        return score, None   # MLP has no hidden state

    def select_action(self, state, evaluate=False, hidden=None):
        s = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        action, new_hidden = self._ddpm_sample(s, hidden)
        return action.cpu().numpy()[0], new_hidden

    # ── RSSM actor loss (eq. 14-16 of paper) ─────────────────
    def _rssm_loss(self, obs_batch, is_weights=None):
        """
        Reverse Sampling Score Matching loss (RSSM, Theorem 3.2).

        Step 1: Sample K actions from current policy for each obs
        Step 2: Evaluate Q-values, compute softmax weights w_k
                w_k = softmax_K(Q(s, ã_k) / α)   [logsumexp for stability]
                This is the reweighting from eq. 13-14 of the paper.
                [ASSUMPTION: logsumexp over K=16 actions approximates partition fn Z]
        Step 3: Sample random diffusion timestep t
        Step 4: Corrupt each ã_0 → ã_t via forward diffusion
        Step 5: Compute reweighted denoising score matching loss

        [ASSUMPTION: ε-parameterisation (DDPM) numerically equivalent
         to score parameterisation in paper eq. 16]
        """
        B = obs_batch.shape[0]
        K = N_SAMPLE_ACTIONS

        # Step 1: sample K actions per state from current policy
        # obs_rep: (B*K, obs_dim)
        obs_rep = obs_batch.unsqueeze(1).expand(-1, K, -1).reshape(B*K, -1)
        with torch.no_grad():
            a0_bk, _ = self._ddpm_sample(obs_rep)   # (B*K, act_dim)
        a0_bk = a0_bk.detach()

        # Step 2: Q-reweighting (eq. 13-14)
        with torch.no_grad():
            q1, q2 = self.critic(obs_rep, a0_bk)    # (B*K, 1)
            q_min  = torch.min(q1, q2)              # (B*K, 1)
            q_min  = q_min.reshape(B, K)            # (B, K)
            # logsumexp trick for numerical stability (paper §4.2)
            log_w  = q_min / self.alpha
            log_w  = log_w - log_w.logsumexp(dim=1, keepdim=True)  # (B, K) normalised log-weights
            w      = log_w.exp()                    # (B, K) softmax weights

        # Step 3: sample timestep t uniformly
        t_idx = torch.randint(0, self.T, (B*K,), device=DEVICE)  # (B*K,)

        # Step 4: forward diffusion  ã_t = √ᾱ_t ã_0 + √(1-ᾱ_t) ε
        sqrt_ab   = self.schedule['sqrt_ab'][t_idx].unsqueeze(1)    # (B*K, 1)
        sqrt_1mab = self.schedule['sqrt_1mab'][t_idx].unsqueeze(1)  # (B*K, 1)
        noise     = torch.randn_like(a0_bk)                        # ε ~ N(0,I)

        # Inverse-rescale ã_0 back to pre-tanh space for diffusion
        a0_raw  = (a0_bk - self.act_bias) / self.act_scale
        a0_raw  = a0_raw.clamp(-0.999, 0.999).atanh()  # pre-tanh
        at      = sqrt_ab * a0_raw + sqrt_1mab * noise  # (B*K, act_dim)

        # Step 5: score network prediction
        # score_pred = self.score_net(obs_rep, at, t_idx)  # (B*K, act_dim)
        score_pred, _ = self._score_forward(obs_rep, at, t_idx)  # (B*K, act_dim)

        # ε-prediction target: the network should predict the noise ε
        # Loss per sample: ||s_θ(ã_t, s, t) − ε||²
        per_dim_loss = (score_pred - noise).pow(2).sum(dim=-1)  # (B*K,)
        per_dim_loss = per_dim_loss.reshape(B, K)               # (B, K)

        # Reweight by softmax Q-weights  w_k (eq. 14-16)
        weighted_loss = (w * per_dim_loss).sum(dim=1)           # (B,)

        # Optional IS weighting for PER variant
        if is_weights is not None:
            weighted_loss = weighted_loss * is_weights

        return weighted_loss.mean()

    # ── Critic update ─────────────────────────────────────────
    def _update_critic(self, s, a, r, ns, mask, is_weights=None):
        """
        Standard SAC twin-Q Bellman update (eq. 10-11).
        log π(ã'|s') approximated by Gaussian log-prob of
        the final denoised action (paper §4, 'log likelihood computation').
        [ASSUMPTION: low-stochasticity approximation of log π via additive Gaussian]
        """
        with torch.no_grad():
            na, _   = self._ddpm_sample(ns)
            # Log-prob approximation: treat final step noise as Gaussian
            # Paper §4: 'we approximate log prob of policy with additive Gaussian'
            log_pi  = -0.5 * ((na - ns[:, :self.act_dim]) ** 2).sum(-1, keepdim=True)
            q1n, q2n = self.critic_tgt(ns, na)
            target_q = r + mask * GAMMA * (torch.min(q1n, q2n) - self.alpha * log_pi)

        q1, q2 = self.critic(s, a)
        if is_weights is not None:
            q1_loss = (is_weights * (q1 - target_q).pow(2).squeeze()).mean()
            q2_loss = (is_weights * (q2 - target_q).pow(2).squeeze()).mean()
        else:
            q1_loss = F.mse_loss(q1, target_q)
            q2_loss = F.mse_loss(q2, target_q)

        q_loss = q1_loss + q2_loss
        self.critic_opt.zero_grad()
        q_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_opt.step()

        # TD errors for PER priority update
        with torch.no_grad():
            td_err = ((q1 + q2) / 2 - target_q).abs().squeeze().cpu().numpy()
        return (q1_loss + q2_loss).item(), td_err

    # ── Alpha update ──────────────────────────────────────────
    def _update_alpha(self, obs):
        with torch.no_grad():
            na, _ = self._ddpm_sample(obs)
            log_pi = -0.5 * ((na - obs[:, :self.act_dim]) ** 2).sum(-1, keepdim=True)

        target_entropy = self._current_target_entropy()
        alpha_loss = -(self.log_alpha * (log_pi + target_entropy).detach()).mean()
        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()
        self.alpha = self.log_alpha.exp().item()
        return alpha_loss.item()

    def _current_target_entropy(self):
        """Override in AdaptiveEntropy subclass."""
        return self.target_entropy  # [ASSUMPTION: fixed -|A|]

    # ── Main update step ─────────────────────────────────────
    def update(self, memory):
        s, a, r, ns, d = memory.sample(BATCH_SIZE)
        # s  = torch.FloatTensor(s).to(DEVICE)
        # a  = torch.FloatTensor(a).to(DEVICE)
        # r  = torch.FloatTensor(r).to(DEVICE).unsqueeze(1)
        # ns = torch.FloatTensor(ns).to(DEVICE)
        # mask = torch.FloatTensor(d).to(DEVICE).unsqueeze(1)
        s    = torch.from_numpy(s).to(DEVICE, non_blocking=True)
        a    = torch.from_numpy(a).to(DEVICE, non_blocking=True)
        r    = torch.from_numpy(r).to(DEVICE, non_blocking=True).unsqueeze(1)
        ns   = torch.from_numpy(ns).to(DEVICE, non_blocking=True)
        mask = torch.from_numpy(d).to(DEVICE, non_blocking=True).unsqueeze(1)
        is_weights = None

        q_loss, td_err = self._update_critic(s, a, r, ns, mask, is_weights)

        actor_loss = self._rssm_loss(s, is_weights)
        self.score_opt.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.score_net.parameters(), 1.0)
        self.score_opt.step()

        alpha_loss = self._update_alpha(s)

        if self.updates % TARGET_UPDATE_INTERVAL == 0:
            soft_update(self.critic_tgt, self.critic, TAU)
        self.updates += 1

        return q_loss, actor_loss.item(), alpha_loss, self.alpha, td_err


# ══════════════════════════════════════════════════════════════
#  V1 — LSTM SDAC  [RELAXED: Markov]
# ══════════════════════════════════════════════════════════════

class LSTMSDACAgent(SDACAgent):
    """
    [RELAXED: Markov assumption]
    Score network replaced with LSTM variant.
    Hidden state carries history across timesteps.
    RSSM loss and critic are unchanged.
    """
    def _make_score_net(self, obs_dim, act_dim):
        return ScoreNetworkLSTM(obs_dim, act_dim, HIDDEN_SIZE)

    def _score_forward(self, obs, noisy_action, t, hidden=None):
        score, new_hidden = self.score_net(obs, noisy_action, t, hidden)
        return score, new_hidden


# ══════════════════════════════════════════════════════════════
#  V2 — PER SDAC  [RELAXED: i.i.d. replay]
# ══════════════════════════════════════════════════════════════

class PERSDACAgent(SDACAgent):
    """
    [RELAXED: i.i.d. replay]
    Uses PrioritizedReplayMemory.
    TD errors from critic update used to update priorities.
    IS weights correct sampling bias.
    """
    def update(self, memory):  # memory is PrioritizedReplayMemory
        s, a, r, ns, d, idxs, is_weights = memory.sample(BATCH_SIZE)
        s  = torch.FloatTensor(s).to(DEVICE)
        a  = torch.FloatTensor(a).to(DEVICE)
        r  = torch.FloatTensor(r).to(DEVICE).unsqueeze(1)
        ns = torch.FloatTensor(ns).to(DEVICE)
        mask = torch.FloatTensor(d).to(DEVICE).unsqueeze(1)

        q_loss, td_err = self._update_critic(s, a, r, ns, mask, is_weights)
        memory.update_priorities(idxs, td_err)

        actor_loss = self._rssm_loss(s, is_weights)
        self.score_opt.zero_grad()
        actor_loss.backward()
        nn.utils.clip_grad_norm_(self.score_net.parameters(), 1.0)
        self.score_opt.step()

        alpha_loss = self._update_alpha(s)

        if self.updates % TARGET_UPDATE_INTERVAL == 0:
            soft_update(self.critic_tgt, self.critic, TAU)
        self.updates += 1

        return q_loss, actor_loss.item(), alpha_loss, self.alpha, td_err


# ══════════════════════════════════════════════════════════════
#  V3 — Adaptive Entropy Target  [RELAXED: fixed H=-|A|]
# ══════════════════════════════════════════════════════════════

class AdaptiveEntropySDACAgent(SDACAgent):
    """
    [RELAXED: fixed entropy target]
    Entropy target annealed: -|A|/2 (first 50%) → -|A| (second 50%).
    Same rationale as SAC V3: more exploration early, tighten late.
    """
    def __init__(self, obs_dim, act_dim, action_space, schedule, total_steps):
        super().__init__(obs_dim, act_dim, action_space, schedule)
        self.act_dim_val  = act_dim
        self.total_steps  = total_steps
        self.env_step_cnt = 0

    def _current_target_entropy(self):
        frac = self.env_step_cnt / self.total_steps
        if frac < 0.5:
            return -self.act_dim_val / 2.0   # more exploratory
        return -float(self.act_dim_val)       # paper default

    def update(self, memory):
        self.env_step_cnt += 1
        return super().update(memory)


# ══════════════════════════════════════════════════════════════
#  LIVE PLOTTER  (same aesthetic as previous notebooks)
# ══════════════════════════════════════════════════════════════

class LivePlotter:
    BG='#0d1117'; PANEL='#161b22'; BORDER='#30363d'
    GRID='#21262d'; TEXT='#f0f6fc'; MUTED='#8b949e'
    BLUE='#58a6ff'; ORANGE='#ffa657'; PURPLE='#d2a8ff'
    GREEN='#3fb950'; RED='#ff7b72'; YELLOW='#e3b341'; CYAN='#79c0ff'

    def __init__(self, variant_name, relaxed, env_id):
        self.name    = variant_name
        self.relaxed = relaxed
        self.env_id  = env_id
        self.steps=[]; self.returns=[]; self.ret_smooth=[]
        self.q_losses=[]; self.actor_losses=[]; self.alphas=[]
        self.recent  = deque(maxlen=20)
        self.last_frame = None

    def _smooth(self, y, w=15):
        if len(y) < w: return np.array(y)
        return np.convolve(y, np.ones(w)/w, mode='valid')

    def _ax(self, ax):
        ax.set_facecolor(self.PANEL)
        ax.grid(True, color=self.GRID, lw=0.5, ls='--', alpha=0.7)
        for sp in ax.spines.values(): sp.set_color(self.BORDER)
        ax.tick_params(colors=self.MUTED, labelsize=6.5)

    def _plot(self, ax, y, color, title):
        ax.cla(); self._ax(ax)
        s = self.steps
        ax.plot(s, y, color=color, lw=0.7, alpha=0.3)
        sm = self._smooth(y)
        if len(sm):
            ax.plot(s[max(0,len(s)-len(sm)):], sm, color=color, lw=1.8)
        ax.set_title(title, color=self.TEXT, fontsize=7, fontweight='bold', pad=3)
        ax.set_xlabel('Steps', color=self.MUTED, fontsize=6)

    def update(self, step, ep_return, q_loss, actor_loss, alpha, frame=None):
        self.steps.append(step)
        self.returns.append(ep_return)
        self.recent.append(ep_return)
        self.ret_smooth.append(float(np.mean(self.recent)))
        self.q_losses.append(q_loss)
        self.actor_losses.append(actor_loss)
        self.alphas.append(alpha)
        if frame is not None: self.last_frame = frame

        fig = plt.figure(figsize=(20, 8), facecolor=self.BG)
        gs  = gridspec.GridSpec(2, 5, figure=fig, hspace=0.55, wspace=0.38,
                                left=0.05, right=0.97, top=0.90, bottom=0.10)
        ax_fr  = fig.add_subplot(gs[:, 0])
        ax_ret = fig.add_subplot(gs[0, 1:4])
        ax_alp = fig.add_subplot(gs[0, 4])
        ax_q   = fig.add_subplot(gs[1, 1])
        ax_act = fig.add_subplot(gs[1, 2])
        ax_buf = fig.add_subplot(gs[1, 3])
        ax_info= fig.add_subplot(gs[1, 4])

        ax_fr.axis('off'); ax_fr.set_facecolor('#000000')
        for ax in [ax_ret, ax_alp, ax_q, ax_act, ax_buf, ax_info]:
            self._ax(ax)

        # Frame
        if self.last_frame is not None:
            ax_fr.imshow(self.last_frame, aspect='auto')
        else:
            ax_fr.text(0.5, 0.5, 'Waiting…', ha='center', va='center',
                       color=self.MUTED, fontsize=10, transform=ax_fr.transAxes)
        ax_fr.set_title(f'{self.name}\n{self.env_id}\nStep {step:,} | Ret {ep_return:.1f}',
                        color=self.TEXT, fontsize=7.5, fontweight='bold', pad=4)

        # Return
        ax_ret.plot(self.steps, self.returns,    color=self.BLUE, lw=0.7, alpha=0.3, label='Raw')
        ax_ret.plot(self.steps, self.ret_smooth, color=self.BLUE, lw=2.0, label='Smooth-20')
        ax_ret.set_title(f'Return — {self.name}  |  [RELAXED: {self.relaxed}]',
                         color=self.TEXT, fontsize=7.5, fontweight='bold', pad=4)
        ax_ret.set_ylabel('Return', color=self.MUTED, fontsize=7)
        ax_ret.legend(fontsize=7, facecolor=self.PANEL, labelcolor=self.MUTED,
                      framealpha=0.6, edgecolor=self.BORDER)

        self._plot(ax_q,   self.q_losses,     self.ORANGE, 'Q Loss  [ASSUMPTION: Markov Q(s,a)]')
        self._plot(ax_act, self.actor_losses,  self.PURPLE, 'RSSM Actor Loss  (eq. 14-16)')
        self._plot(ax_alp, self.alphas,        self.GREEN,  'Alpha  [ASSUMPTION: auto-temp H=-|A|]')

        # Buffer fill
        fill = [min(st/REPLAY_SIZE*100, 100) for st in self.steps]
        ax_buf.plot(self.steps, fill, color=self.YELLOW, lw=1.8)
        ax_buf.set_title('Replay Buffer Fill  [ASSUMPTION: off-policy i.i.d.]',
                          color=self.TEXT, fontsize=7, fontweight='bold', pad=3)
        ax_buf.set_xlabel('Steps', color=self.MUTED, fontsize=6)

        # Info panel
        ax_info.axis('off')
        ax_info.text(0.05, 0.85, 'SDAC Algorithm 1', color=self.TEXT,
                     fontsize=8, fontweight='bold', transform=ax_info.transAxes)
        info = (
            'Score net: ε-prediction\n'
            f'Diffusion steps T={N_DIFFUSION_STEPS}\n'
            f'Sample actions K={N_SAMPLE_ACTIONS}\n'
            'Reweight: softmax(Q/α)\n'
            'Critic: twin-Q SAC-style'
        )
        ax_info.text(0.05, 0.55, info, color=self.MUTED, fontsize=7.5,
                     transform=ax_info.transAxes, verticalalignment='top',
                     fontfamily='monospace')

        fig.suptitle(f'SDAC — {self.env_id}  |  {self.name}  |  Steps: {step:,}',
                     color=self.TEXT, fontsize=10, fontweight='bold', y=0.97)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=90, bbox_inches='tight', facecolor=self.BG)
        buf.seek(0); plt.close(fig)
        clear_output(wait=True); display(Image.open(buf))


# ══════════════════════════════════════════════════════════════
#  TRAINING LOOP  (shared by all tasks and variants)
# ══════════════════════════════════════════════════════════════

def run_variant(agent, memory, variant_name, relaxed, env_id, total_steps=TOTAL_STEPS):
    env        = gym.make(env_id)
    render_env = gym.make(env_id, render_mode='rgb_array')
    plotter    = LivePlotter(variant_name, relaxed, env_id)

    history = dict(steps=[], returns=[], q_losses=[], actor_losses=[],
                   alphas=[], wall_times=[])

    log_q, log_act, log_alph = [], [], []
    ep_returns = []
    current_frame = None
    hidden = None
    start  = time.time()

    obs, _ = env.reset(seed=SEED)
    ep_ret = 0.0
    t      = 0

    print(f"\n{'='*65}")
    print(f"  {variant_name}  |  {env_id}  |  Relaxed: {relaxed}")
    print(f"{'='*65}")

    while t < total_steps:
        # Action selection
        if t < START_STEPS:
            action = env.action_space.sample()
            hidden = None
        else:
            action, hidden = agent.select_action(obs, hidden=hidden)

        next_obs, reward, terminated, truncated, _ = env.step(action)
        done    = terminated or truncated
        ep_ret += reward
        t      += 1
        mask    = 0.0 if terminated else 1.0
        memory.push(obs, action, reward, next_obs, mask)
        obs = next_obs

        if done:
            ep_returns.append(ep_ret)
            ep_ret = 0.0
            hidden = None
            obs, _ = env.reset()

        # Gradient update
        if len(memory) > BATCH_SIZE:
            q_l, act_l, alph_l, alph, td_err = agent.update(memory)
            log_q.append(q_l); log_act.append(act_l); log_alph.append(alph)

        # Render
        if t % RENDER_EVERY == 0 or t == 1:
            try:
                r_obs, _ = render_env.reset()
                r_hidden = None; latest = None
                for _ in range(1000):
                    r_act, r_hidden = agent.select_action(r_obs, evaluate=True,
                                                          hidden=r_hidden)
                    r_obs, _, rt, ru, _ = render_env.step(r_act)
                    f = render_env.render()
                    if f is not None: latest = f
                    if rt or ru: break
                if latest is not None: current_frame = latest
            except Exception:
                pass

        # Dashboard
        if t % PLOT_EVERY == 0 and log_q:
            safe_ret = float(np.mean(ep_returns[-20:])) if ep_returns else 0.0
            plotter.update(t, safe_ret,
                           float(np.mean(log_q)),
                           float(np.mean(log_act)),
                           float(np.mean(log_alph)),
                           frame=current_frame)
            history['steps'].append(t)
            history['returns'].append(safe_ret)
            history['q_losses'].append(float(np.mean(log_q)))
            history['actor_losses'].append(float(np.mean(log_act)))
            history['alphas'].append(float(np.mean(log_alph)))
            history['wall_times'].append(time.time() - start)
            log_q, log_act, log_alph = [], [], []

            if t % (PLOT_EVERY * 10) == 0:
                sps = int(t / (time.time() - start))
                print(f"  step={t:,}  ret={safe_ret:.1f}  "
                      f"q={history['q_losses'][-1]:.4f}  "
                      f"α={history['alphas'][-1]:.4f}  {sps} SPS")

    env.close(); render_env.close()
    history['total_wall_time'] = time.time() - start
    print(f"\n✓ {variant_name} done in {history['total_wall_time']/60:.1f} min")
    return history


# ══════════════════════════════════════════════════════════════
#  PER-TASK COMPARISON DASHBOARD
# ══════════════════════════════════════════════════════════════

COLORS = {
    'V0 Baseline':   '#58a6ff',
    'V1 LSTM':       '#ffa657',
    'V2 PER':        '#3fb950',
    'V3 Adaptive H': '#d2a8ff',
}

def smooth(y, w=20):
    if len(y) < w: return np.array(y)
    return np.convolve(y, np.ones(w)/w, mode='valid')


def plot_task_comparison(all_histories, env_id, save_path=None):
    BG=LivePlotter.BG; PANEL=LivePlotter.PANEL; BORDER=LivePlotter.BORDER
    GRID=LivePlotter.GRID; TEXT=LivePlotter.TEXT; MUTED=LivePlotter.MUTED
    THRESHOLD = 500.0   # task-agnostic return threshold for convergence bar

    fig, axes = plt.subplots(2, 3, figsize=(21, 10), facecolor=BG)
    (ax_ret, ax_wall, ax_q,
     ax_act, ax_alp, ax_conv) = axes.flatten()

    titles = ['Episode Return', 'Return vs Wall-Clock',
              'Q Loss', 'Actor (RSSM) Loss', 'Alpha', 'Convergence Speed']
    for ax, title in zip(axes.flatten(), titles):
        ax.set_facecolor(PANEL)
        ax.grid(True, color=GRID, lw=0.5, ls='--', alpha=0.7)
        for sp in ax.spines.values(): sp.set_color(BORDER)
        ax.tick_params(colors=MUTED, labelsize=8)
        ax.set_title(title, color=TEXT, fontsize=9, fontweight='bold', pad=5)

    convergence_data = []

    for name, history in all_histories.items():
        color = COLORS.get(name, '#ffffff')
        s  = np.array(history['steps'])
        r  = np.array(history['returns'])
        wt = np.array(history['wall_times'])
        sr = smooth(r)
        s_sm = s[max(0, len(s)-len(sr)):]

        ax_ret.plot(s, r, color=color, lw=0.6, alpha=0.2)
        ax_ret.plot(s_sm, sr, color=color, lw=2.2, label=name)

        ax_wall.plot(wt, r, color=color, lw=0.6, alpha=0.2)
        ax_wall.plot(wt[max(0,len(wt)-len(sr)):], sr, color=color, lw=2.2, label=name)

        sq = smooth(history['q_losses'])
        ax_q.plot(s[max(0,len(s)-len(sq)):], sq, color=color, lw=1.8, label=name)

        sa = smooth(history['actor_losses'])
        ax_act.plot(s[max(0,len(s)-len(sa)):], sa, color=color, lw=1.8, label=name)

        salp = smooth(history['alphas'])
        ax_alp.plot(s[max(0,len(s)-len(salp)):], salp, color=color, lw=1.8, label=name)

        above = np.where(sr > THRESHOLD)[0]
        conv  = int(s_sm[above[0]]) if len(above) > 0 else None
        convergence_data.append((name, conv, history['total_wall_time'], color))

    # Convergence bar
    names  = [d[0] for d in convergence_data]
    c_vals = [d[1] if d[1] is not None else TOTAL_STEPS for d in convergence_data]
    c_cols = [d[3] for d in convergence_data]
    bars   = ax_conv.bar(names, c_vals, color=c_cols, alpha=0.8, edgecolor=BORDER)
    ax_conv.axhline(TOTAL_STEPS, color='#ff7b72', lw=1.2, ls='--', alpha=0.5)
    for bar, v in zip(bars, c_vals):
        label = f'{v:,}' if v < TOTAL_STEPS else 'N/A'
        ax_conv.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                     label, ha='center', va='bottom', color=TEXT, fontsize=7)
    ax_conv.set_ylabel(f'Steps to return>{THRESHOLD}', color=MUTED, fontsize=8)
    ax_conv.tick_params(axis='x', colors=TEXT, labelsize=7)

    for ax in [ax_ret, ax_wall, ax_q, ax_act, ax_alp]:
        ax.legend(fontsize=7.5, facecolor=PANEL, labelcolor=MUTED,
                  framealpha=0.7, edgecolor=BORDER)
    ax_ret.set_xlabel('Steps', color=MUTED, fontsize=8)
    ax_wall.set_xlabel('Wall-clock (s)', color=MUTED, fontsize=8)

    annot = (
        'V0 Baseline:   All SDAC assumptions intact (Markov, i.i.d., fixed H=-|A|)\n'
        'V1 LSTM:       [RELAXED Markov] LSTM score network carries history\n'
        'V2 PER:        [RELAXED i.i.d.] Prioritized replay by |TD error|\n'
        'V3 Adaptive H: [RELAXED fixed H] Entropy target -|A|/2 → -|A|'
    )
    fig.text(0.02, 0.01, annot, color=MUTED, fontsize=7.5, fontfamily='monospace',
             bbox=dict(facecolor=PANEL, edgecolor=BORDER, alpha=0.9, pad=6))

    fig.suptitle(f'SDAC Assumption Relaxation — {env_id}  ({TOTAL_STEPS//1000}k steps each)',
                 color=TEXT, fontsize=12, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0.07, 1, 0.97])

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight', facecolor=BG)
    buf.seek(0); plt.close(fig)
    img = Image.open(buf)
    display(img)
    if save_path:
        img.save(save_path)
        print(f'Saved → {save_path}')
    return convergence_data


def make_schedule():
    return make_ddpm_schedule(N_DIFFUSION_STEPS, BETA_MIN, BETA_MAX, DEVICE)


print('\n✓ Cell 1 complete — all shared code loaded.')
print(f'  ENV detection will happen per-task cell.')
print(f'  SDAC: T={N_DIFFUSION_STEPS} steps, K={N_SAMPLE_ACTIONS} actions, '
      f'hidden={HIDDEN_SIZE}, steps/variant={TOTAL_STEPS:,}')


def save_evaluation_video(agent, env_id, save_path, n_episodes=3, max_steps=1000):
    """
    Runs n_episodes of greedy evaluation and saves an mp4 video.
    Uses imageio which is already installed.
    """
    import imageio

    print(f"\n🎬 Recording video: {save_path}")
    eval_env   = gym.make(env_id, render_mode='rgb_array')
    frames     = []
    ep_returns = []

    for ep in range(n_episodes):
        obs, _ = eval_env.reset(seed=ep * 13)
        hidden  = None
        ep_ret  = 0.0
        for _ in range(max_steps):
            action, hidden = agent.select_action(obs, evaluate=True, hidden=hidden)
            obs, reward, terminated, truncated, _ = eval_env.step(action)
            ep_ret += reward
            frame = eval_env.render()
            if frame is not None:
                frames.append(frame)
            if terminated or truncated:
                break
        ep_returns.append(ep_ret)
        print(f"  Episode {ep+1}: return = {ep_ret:.1f}")

    eval_env.close()

    if frames:
        imageio.mimsave(save_path, frames, fps=30, macro_block_size=1)
        print(f"  ✓ Saved {len(frames)} frames → {save_path}")
        print(f"  Mean return: {np.mean(ep_returns):.1f}")
    else:
        print("  ✗ No frames captured.")

# ══════════════════════════════════════════════════════════════
#  V4 — Adaptive Diffusion Steps  [RELAXED: fixed T]
#
#  Fixed T=5 assumes all stages of training need the same
#  denoising depth. We relax this by annealing T:
#    Phase 1 (0→33%):  T=10  (more expressive early exploration)
#    Phase 2 (33→66%): T=5   (paper default)
#    Phase 3 (66→100%): T=3  (faster inference when near-optimal)
#
#  Only _ddpm_sample changes — RSSM loss, critic, alpha unchanged.
# ══════════════════════════════════════════════════════════════

class AdaptiveTSDACAgent(SDACAgent):
    """
    [RELAXED: fixed diffusion step count T]
    T anneals 10 → 5 → 3 over training.
    """
    def __init__(self, obs_dim, act_dim, action_space, schedule, total_steps):
        super().__init__(obs_dim, act_dim, action_space, schedule)
        self.total_steps  = total_steps
        self.env_step_cnt = 0

    def _current_T(self):
        frac = self.env_step_cnt / self.total_steps
        if frac < 0.33:
            return 10
        elif frac < 0.66:
            return 5
        else:
            return 3

    @torch.no_grad()
    def _ddpm_sample(self, obs, hidden=None):
        """
        Same DDPM reverse process as base, but T is dynamic.
        [RELAXED: T now depends on training progress]
        """
        T_now = self._current_T()
        B     = obs.shape[0]
        at    = torch.randn(B, self.act_dim, device=DEVICE)

        # Recompute schedule for current T on the fly
        # We sub-sample the original schedule indices uniformly
        indices = torch.linspace(0, self.T - 1, T_now,
                                 device=DEVICE).long()

        for i, t_idx in enumerate(reversed(indices.tolist())):
            t_tensor  = torch.full((B,), t_idx, device=DEVICE,
                                   dtype=torch.long)
            score, hidden = self._score_forward(obs, at, t_tensor, hidden)

            beta_t    = self.schedule['betas'][t_idx]
            alpha_t   = self.schedule['alphas'][t_idx]
            sqrt_1mab = self.schedule['sqrt_1mab'][t_idx]
            coef      = beta_t / sqrt_1mab
            mean      = (1.0 / alpha_t.sqrt()) * (at - coef * score)
            at = mean + (beta_t.sqrt() * torch.randn_like(at)
                         if i < T_now - 1 else torch.zeros_like(at))

        return torch.tanh(at) * self.act_scale + self.act_bias, hidden

    def update(self, memory):
        self.env_step_cnt += 1
        return super().update(memory)


# ══════════════════════════════════════════════════════════════
#  V5 — Recency-Weighted Replay  [RELAXED: stationarity of
#       behavior policy]
#
#  Uniform replay treats a transition from step 100 (random
#  policy) identically to one from step 900k (near-optimal).
#  This violates the implicit stationarity assumption.
#  We relax it by exponentially down-weighting older transitions.
#
#  Weight of transition stored at time t_stored:
#    w(t_stored) ∝ exp(λ · (t_stored - t_oldest) / buffer_size)
#  λ controls recency bias strength (λ=0 → uniform, λ=5 → strong recency)
#
#  Implementation: each slot stores a timestamp; sampling uses
#  weighted random choice via numpy (O(N) but fine for 50k buffer).
#  For 1M buffer use alias method or a separate priority tree.
# ══════════════════════════════════════════════════════════════

class RecencyReplayMemory:
    """
    [RELAXED: stationarity of behavior policy]
    Transitions sampled proportional to recency via exponential weighting.
    Distinct from PER: prioritizes *when* collected, not TD error magnitude.
    """
    def __init__(self, capacity, seed, lam=3.0):
        random.seed(seed)
        np.random.seed(seed)
        self.capacity  = capacity
        self.lam       = lam       # recency bias strength
        self.buf       = []
        self.pos       = 0
        self.timestamps = np.zeros(capacity, dtype=np.int64)
        self.t_global   = 0

    def push(self, s, a, r, ns, done):
        if len(self.buf) < self.capacity:
            self.buf.append(None)
        self.buf[self.pos]       = (s, a, r, ns, done)
        self.timestamps[self.pos] = self.t_global
        self.pos      = (self.pos + 1) % self.capacity
        self.t_global += 1

    def sample(self, n):
        N   = len(self.buf)
        ts  = self.timestamps[:N]
        # Exponential recency weights
        # Normalise so oldest=0, newest=1 then apply exp(λ·x)
        t_min = ts.min(); t_max = ts.max()
        if t_max > t_min:
            norm    = (ts - t_min) / (t_max - t_min)
            weights = np.exp(self.lam * norm)
        else:
            weights = np.ones(N)
        weights /= weights.sum()

        indices = np.random.choice(N, size=n, replace=False, p=weights)
        batch   = [self.buf[i] for i in indices]
        s, a, r, ns, d = map(lambda x: np.array(x, dtype=np.float32),
                              zip(*batch))
        return s, a, r, ns, d

    def __len__(self):
        return len(self.buf)


# V5 agent — identical to base SDACAgent, only memory class differs
# No agent subclass needed: just pass RecencyReplayMemory to run_variant.
# We define a thin alias for naming consistency in the loop.
class RecencySDACAgent(SDACAgent):
    """
    [RELAXED: stationarity of behavior policy]
    Agent is identical to baseline — relaxation is entirely in the
    RecencyReplayMemory. Defined as subclass only for clear variant labelling.
    """
    pass


# ══════════════════════════════════════════════════════════════
#  V6 — State-Dependent Temperature α(s)  [RELAXED: scalar α]
#
#  Baseline uses a single scalar α shared across all states.
#  This assumes the optimal entropy level is uniform over state space.
#  We relax this with a learned α(s) network:
#    log α(s) = MLP(s)  →  α(s) = exp(log α(s))
#
#  Training objective (per-state extension of SAC auto-α):
#    L_α = E_s[ −α(s) · (log π(ã|s) + H_target) ]
#  Gradient flows through α(s) w.r.t. the MLP parameters.
#
#  [PORT-4]: Scalar α is standard in SAC and SDAC paper.
#  State-dependent α is a known extension (e.g. LYSAC, 2021)
#  applied here to the diffusion policy setting.
# ══════════════════════════════════════════════════════════════

class AlphaNetwork(nn.Module):
    """
    Small MLP that outputs log α given state s.
    Output clamped to [-5, 2] → α in [exp(-5), exp(2)] ≈ [0.007, 7.4]
    """
    def __init__(self, obs_dim, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight, gain=0.1)
                nn.init.zeros_(m.bias)

    def forward(self, obs):
        # Returns α(s) — one value per state in batch
        log_alpha = self.net(obs).clamp(-5.0, 2.0)
        return log_alpha.exp()          # (B, 1)

    def log_alpha(self, obs):
        return self.net(obs).clamp(-5.0, 2.0)  # (B, 1)


class StateDepAlphaSDACAgent(SDACAgent):
    """
    [RELAXED: scalar temperature]
    α is now a function of state: α(s) predicted by AlphaNetwork.
    Critic, score network, DDPM sampling all unchanged.
    Only the alpha update and its usage in RSSM/critic change.
    """
    def __init__(self, obs_dim, act_dim, action_space, schedule):
        super().__init__(obs_dim, act_dim, action_space, schedule)
        # Replace scalar alpha machinery with network
        self.alpha_net = AlphaNetwork(obs_dim).to(DEVICE)
        self.alpha_opt = Adam(self.alpha_net.parameters(), lr=LR_ALPHA)
        # Keep self.alpha as a running mean for logging only
        self.alpha = ALPHA_INIT

    def _get_alpha(self, obs):
        """Returns per-state α(s) tensor of shape (B, 1)."""
        return self.alpha_net(obs)

    def _rssm_loss(self, obs_batch, is_weights=None):
        """
        [RELAXED: scalar α → α(s)]
        Q-reweighting now uses per-state temperature.
        """
        B = obs_batch.shape[0]
        K = N_SAMPLE_ACTIONS

        obs_rep = obs_batch.unsqueeze(1).expand(-1, K, -1).reshape(B*K, -1)
        with torch.no_grad():
            a0_bk, _ = self._ddpm_sample(obs_rep)
        a0_bk = a0_bk.detach()

        with torch.no_grad():
            q1, q2  = self.critic(obs_rep, a0_bk)
            q_min   = torch.min(q1, q2).reshape(B, K)

            # Per-state α: shape (B, 1), broadcast over K
            alpha_s = self._get_alpha(obs_batch)          # (B, 1)
            log_w   = q_min / alpha_s                     # (B, K)
            log_w   = log_w - log_w.logsumexp(dim=1, keepdim=True)
            w       = log_w.exp()

        t_idx     = torch.randint(0, self.T, (B*K,), device=DEVICE)
        sqrt_ab   = self.schedule['sqrt_ab'][t_idx].unsqueeze(1)
        sqrt_1mab = self.schedule['sqrt_1mab'][t_idx].unsqueeze(1)
        noise     = torch.randn_like(a0_bk)
        a0_raw    = (a0_bk - self.act_bias) / self.act_scale
        a0_raw    = a0_raw.clamp(-0.999, 0.999).atanh()
        at        = sqrt_ab * a0_raw + sqrt_1mab * noise

        score_pred, _ = self._score_forward(obs_rep, at, t_idx)
        per_dim_loss  = (score_pred - noise).pow(2).sum(dim=-1).reshape(B, K)
        weighted_loss = (w * per_dim_loss).sum(dim=1)

        if is_weights is not None:
            weighted_loss = weighted_loss * is_weights

        return weighted_loss.mean()

    def _update_critic(self, s, a, r, ns, mask, is_weights=None):
        """
        [RELAXED: scalar α → α(s') in Bellman target]
        """
        with torch.no_grad():
            na, _    = self._ddpm_sample(ns)
            log_pi   = -0.5 * ((na - ns[:, :self.act_dim])**2).sum(-1, keepdim=True)
            alpha_ns = self._get_alpha(ns)               # (B, 1)
            q1n, q2n = self.critic_tgt(ns, na)
            target_q = r + mask * GAMMA * (
                torch.min(q1n, q2n) - alpha_ns * log_pi)

        q1, q2 = self.critic(s, a)
        if is_weights is not None:
            q1_loss = (is_weights * (q1 - target_q).pow(2).squeeze()).mean()
            q2_loss = (is_weights * (q2 - target_q).pow(2).squeeze()).mean()
        else:
            q1_loss = F.mse_loss(q1, target_q)
            q2_loss = F.mse_loss(q2, target_q)

        q_loss = q1_loss + q2_loss
        self.critic_opt.zero_grad()
        q_loss.backward()
        nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_opt.step()

        with torch.no_grad():
            td_err = ((q1 + q2)/2 - target_q).abs().squeeze().cpu().numpy()
        return (q1_loss + q2_loss).item(), td_err

    def _update_alpha(self, obs):
        """
        [RELAXED: scalar α → network α(s)]
        Loss: E_s[ -α(s) · (log π(ã|s) + H_target) ]
        Gradient flows through α_net parameters.
        """
        with torch.no_grad():
            na, _  = self._ddpm_sample(obs)
            log_pi = -0.5 * ((na - obs[:, :self.act_dim])**2).sum(-1, keepdim=True)

        alpha_s    = self._get_alpha(obs)                # (B, 1) — with grad
        alpha_loss = -(alpha_s * (log_pi +
                       self.target_entropy).detach()).mean()

        self.alpha_opt.zero_grad()
        alpha_loss.backward()
        self.alpha_opt.step()

        # Update scalar alpha for logging (mean over batch)
        self.alpha = self._get_alpha(obs).mean().item()
        return alpha_loss.item()


# ── Update COLORS dict to include new variants ────────────────
COLORS.update({
    'V4 Adaptive T':   '#ff6e96',
    'V5 Recency':      '#56d364',
    'V6 State-α':      '#f0e68c',
})

print('✓ V4 / V5 / V6 agent classes loaded.')

In [ ]:
# ============================================================
#  CELL 2 — HalfCheetah-v4  (V4, V5, V6 new relaxations)
#  Assumes hc_histories already exists from previous notebook's
#  Cell 2 (V0 baseline is reused as reference).
#  If starting fresh, V0 will be re-trained here too.
# ============================================================
ENV_HC = 'HalfCheetah-v4'
try:
    _e = gym.make(ENV_HC); _e.reset(); _e.close()
    print(f'✓ {ENV_HC} available')
except Exception as ex:
    print(f'✗ {ENV_HC} not available: {ex}')
    ENV_HC = None

if ENV_HC is not None:
    _probe   = gym.make(ENV_HC)
    _obs_dim = _probe.observation_space.shape[0]
    _act_dim = _probe.action_space.shape[0]
    _act_sp  = _probe.action_space
    _probe.close()

    # Initialise history dict — add V0 baseline if not already present
    if 'hc_histories_new' not in globals():
        hc_histories_new = {}

    _trained_agents_hc = {}

    for vname, AgentCls, MemCls, extra in [
        ('V0 Baseline',  SDACAgent,
         ReplayMemory, {}),
        ('V4 Adaptive T', AdaptiveTSDACAgent,
         ReplayMemory, {'total_steps': TOTAL_STEPS}),
        ('V5 Recency',   RecencySDACAgent,
         RecencyReplayMemory, {}),
        ('V6 State-α',   StateDepAlphaSDACAgent,
         ReplayMemory, {}),
    ]:
        relaxed = {
            'V0 Baseline':   'None (reference)',
            'V4 Adaptive T': 'Fixed T (adaptive 10→5→3)',
            'V5 Recency':    'i.i.d. replay (recency-weighted)',
            'V6 State-α':    'Scalar temperature (state-dep α(s))',
        }[vname]
        print(f'\n▶  {vname}')
        agent = AgentCls(_obs_dim, _act_dim, _act_sp, make_schedule(), **extra)
        mem   = MemCls(REPLAY_SIZE, SEED)
        hc_histories_new[vname] = run_variant(
            agent, mem, vname, relaxed, ENV_HC)
        _trained_agents_hc[vname] = agent

    # Videos
    for vname, agent_obj in _trained_agents_hc.items():
        safe = vname.replace(' ', '_').replace('/', '_')
        save_evaluation_video(
            agent_obj, ENV_HC,
            save_path=f'/kaggle/working/sdac_hc_{safe}_new.mp4')

    print('\n── HalfCheetah new-variant comparison ──')
    hc_conv_new = plot_task_comparison(
        hc_histories_new, ENV_HC,
        save_path='/kaggle/working/sdac_halfcheetah_new_comparison.png')

In [ ]:
ENV_ANT = 'Ant-v4'
try:
    _e = gym.make(ENV_ANT); _e.reset(); _e.close()
    print(f'✓ {ENV_ANT} available')
except Exception as ex:
    print(f'✗ {ENV_ANT} not available: {ex}')
    ENV_ANT = None

if ENV_ANT is not None:
    _probe   = gym.make(ENV_ANT)
    _obs_dim = _probe.observation_space.shape[0]
    _act_dim = _probe.action_space.shape[0]
    _act_sp  = _probe.action_space
    _probe.close()

    # Initialise history dict — add V0 baseline if not already present
    if 'ant_histories_new' not in globals():
        ant_histories_new = {}

    _trained_agents_ant = {}

    for vname, AgentCls, MemCls, extra in [
        ('V0 Baseline',  SDACAgent,
         ReplayMemory, {}),
        ('V4 Adaptive T', AdaptiveTSDACAgent,
         ReplayMemory, {'total_steps': TOTAL_STEPS}),
        ('V5 Recency',   RecencySDACAgent,
         RecencyReplayMemory, {}),
        ('V6 State-α',   StateDepAlphaSDACAgent,
         ReplayMemory, {}),
    ]:
        relaxed = {
            'V0 Baseline':   'None (reference)',
            'V4 Adaptive T': 'Fixed T (adaptive 10→5→3)',
            'V5 Recency':    'i.i.d. replay (recency-weighted)',
            'V6 State-α':    'Scalar temperature (state-dep α(s))',
        }[vname]
        print(f'\n▶  {vname}')
        agent = AgentCls(_obs_dim, _act_dim, _act_sp, make_schedule(), **extra)
        mem   = MemCls(REPLAY_SIZE, SEED)
        ant_histories_new[vname] = run_variant(
            agent, mem, vname, relaxed, ENV_ANT)
        _trained_agents_ant[vname] = agent

    # Videos
    for vname, agent_obj in _trained_agents_ant.items():
        safe = vname.replace(' ', '_').replace('/', '_')
        save_evaluation_video(
            agent_obj, ENV_ANT,
            save_path=f'/kaggle/working/sdac_ant_{safe}_new.mp4')

    print('\n── Ant new-variant comparison ──')
    ant_conv_new = plot_task_comparison(
        ant_histories_new, ENV_ANT,
        save_path='/kaggle/working/sdac_ant_new_comparison.png')

In [ ]:
ENV_HOP = 'Hopper-v4'
try:
    _e = gym.make(ENV_HOP); _e.reset(); _e.close()
    print(f'✓ {ENV_HOP} available')
except Exception as ex:
    print(f'✗ {ENV_HOP} not available: {ex}')
    ENV_HOP = None

if ENV_HOP is not None:
    _probe   = gym.make(ENV_HOP)
    _obs_dim = _probe.observation_space.shape[0]
    _act_dim = _probe.action_space.shape[0]
    _act_sp  = _probe.action_space
    _probe.close()

    # Initialise history dict — add V0 baseline if not already present
    if 'hop_histories_new' not in globals():
        hop_histories_new = {}

    _trained_agents_hop = {}

    for vname, AgentCls, MemCls, extra in [
        ('V0 Baseline',  SDACAgent,
         ReplayMemory, {}),
        ('V4 Adaptive T', AdaptiveTSDACAgent,
         ReplayMemory, {'total_steps': TOTAL_STEPS}),
        ('V5 Recency',   RecencySDACAgent,
         RecencyReplayMemory, {}),
        ('V6 State-α',   StateDepAlphaSDACAgent,
         ReplayMemory, {}),
    ]:
        relaxed = {
            'V0 Baseline':   'None (reference)',
            'V4 Adaptive T': 'Fixed T (adaptive 10→5→3)',
            'V5 Recency':    'i.i.d. replay (recency-weighted)',
            'V6 State-α':    'Scalar temperature (state-dep α(s))',
        }[vname]
        print(f'\n▶  {vname}')
        agent = AgentCls(_obs_dim, _act_dim, _act_sp, make_schedule(), **extra)
        mem   = MemCls(REPLAY_SIZE, SEED)
        hop_histories_new[vname] = run_variant(
            agent, mem, vname, relaxed, ENV_HOP)
        _trained_agents_hop[vname] = agent

    # Videos
    for vname, agent_obj in _trained_agents_hop.items():
        safe = vname.replace(' ', '_').replace('/', '_')
        save_evaluation_video(
            agent_obj, ENV_HOP,
            save_path=f'/kaggle/working/sdac_hopper_{safe}_new.mp4')

    print('\n── Hopper new-variant comparison ──')
    hop_conv_new = plot_task_comparison(
        hop_histories_new, ENV_HOP,
        save_path='/kaggle/working/sdac_hop_new_comparison.png')

In [ ]:
ENV_WLK = 'Walker2d-v4'
try:
    _e = gym.make(ENV_WLK); _e.reset(); _e.close()
    print(f'✓ {ENV_WLK} available')
except Exception as ex:
    print(f'✗ {ENV_WLK} not available: {ex}')
    ENV_WLK = None

if ENV_WLK is not None:
    _probe   = gym.make(ENV_WLK)
    _obs_dim = _probe.observation_space.shape[0]
    _act_dim = _probe.action_space.shape[0]
    _act_sp  = _probe.action_space
    _probe.close()

    # Initialise history dict — add V0 baseline if not already present
    if 'wlk_histories_new' not in globals():
        wlk_histories_new = {}

    _trained_agents_wlk = {}

    for vname, AgentCls, MemCls, extra in [
        ('V0 Baseline',  SDACAgent,
         ReplayMemory, {}),
        ('V4 Adaptive T', AdaptiveTSDACAgent,
         ReplayMemory, {'total_steps': TOTAL_STEPS}),
        ('V5 Recency',   RecencySDACAgent,
         RecencyReplayMemory, {}),
        ('V6 State-α',   StateDepAlphaSDACAgent,
         ReplayMemory, {}),
    ]:
        relaxed = {
            'V0 Baseline':   'None (reference)',
            'V4 Adaptive T': 'Fixed T (adaptive 10→5→3)',
            'V5 Recency':    'i.i.d. replay (recency-weighted)',
            'V6 State-α':    'Scalar temperature (state-dep α(s))',
        }[vname]
        print(f'\n▶  {vname}')
        agent = AgentCls(_obs_dim, _act_dim, _act_sp, make_schedule(), **extra)
        mem   = MemCls(REPLAY_SIZE, SEED)
        wlk_histories_new[vname] = run_variant(
            agent, mem, vname, relaxed, ENV_WLK)
        _trained_agents_wlk[vname] = agent

    # Videos
    for vname, agent_obj in _trained_agents_wlk.items():
        safe = vname.replace(' ', '_').replace('/', '_')
        save_evaluation_video(
            agent_obj, ENV_WLK,
            save_path=f'/kaggle/working/sdac_wlk_{safe}_new.mp4')

    print('\n── Walker2d new-variant comparison ──')
    wlk_conv_new = plot_task_comparison(
        wlk_histories_new, ENV_WLK,
        save_path='/kaggle/working/sdac_wlk_new_comparison.png')

In [ ]:
# ============================================================
#  CELL 6 — Cross-task summary dashboard
#  Shows final smoothed return per variant per task,
#  and timing table.
# ============================================================

import io

# Collect available task results
_available = []
# for name, hist in [('HalfCheetah-v4', globals().get('hc_histories')),
#                    ('Ant-v4',          globals().get('ant_histories')),
#                    ('Hopper-v4',       globals().get('hop_histories')),
#                    ('Walker2d-v4',     globals().get('wlk_histories'))]:
# New variant histories
for name, hist in [('HalfCheetah-v4', globals().get('hc_histories_new')),
                   ('Ant-v4',          globals().get('ant_histories_new')),
                   ('Hopper-v4',       globals().get('hop_histories_new')),
                   ('Walker2d-v4',     globals().get('wlk_histories_new'))]:
    if hist:
        _available.append((name, hist))

if not _available:
    print('No task histories found. Run Cells 2-5 first.')
else:
    BG=LivePlotter.BG; PANEL=LivePlotter.PANEL; BORDER=LivePlotter.BORDER
    GRID=LivePlotter.GRID; TEXT=LivePlotter.TEXT; MUTED=LivePlotter.MUTED

    n_tasks   = len(_available)
    variants  = list(COLORS.keys())
    fig, axes = plt.subplots(2, max(2, n_tasks), figsize=(6*max(2,n_tasks), 10),
                             facecolor=BG)
    if n_tasks == 1:
        axes = axes.reshape(2, 1)

    for col, (task_name, hist) in enumerate(_available):
        ax_top = axes[0][col]
        ax_bot = axes[1][col]
        for ax in [ax_top, ax_bot]:
            ax.set_facecolor(PANEL)
            ax.grid(True, color=GRID, lw=0.5, ls='--', alpha=0.7)
            for sp in ax.spines.values(): sp.set_color(BORDER)
            ax.tick_params(colors=MUTED, labelsize=8)

        # Top: smoothed return curves
        for vname, color in COLORS.items():
            if vname not in hist: continue
            s = np.array(hist[vname]['steps'])
            r = np.array(hist[vname]['returns'])
            sr = smooth(r)
            ax_top.plot(s[max(0,len(s)-len(sr)):], sr,
                        color=color, lw=2.0, label=vname)
        ax_top.set_title(task_name, color=TEXT, fontsize=9, fontweight='bold', pad=5)
        ax_top.set_xlabel('Steps', color=MUTED, fontsize=8)
        ax_top.set_ylabel('Return', color=MUTED, fontsize=8)
        ax_top.legend(fontsize=7, facecolor=PANEL, labelcolor=MUTED,
                      framealpha=0.7, edgecolor=BORDER)

        # Bottom: training time bar chart
        times  = [hist[v]['total_wall_time']/60 for v in variants if v in hist]
        vnames = [v for v in variants if v in hist]
        cols   = [COLORS[v] for v in vnames]
        bars   = ax_bot.bar(vnames, times, color=cols, alpha=0.8, edgecolor=BORDER)
        for bar, t in zip(bars, times):
            ax_bot.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                        f'{t:.1f}m', ha='center', va='bottom', color=TEXT, fontsize=7)
        ax_bot.set_title('Training Time (min)', color=TEXT, fontsize=8,
                         fontweight='bold', pad=5)
        ax_bot.tick_params(axis='x', colors=TEXT, labelsize=6.5)

    fig.suptitle('SDAC Assumption Relaxation — Cross-Task Summary',
                 color=TEXT, fontsize=13, fontweight='bold', y=0.99)
    plt.tight_layout(rect=[0, 0, 1, 0.97])

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110, bbox_inches='tight', facecolor=BG)
    buf.seek(0); plt.close(fig)
    img = Image.open(buf)
    display(img)
    img.save('/kaggle/working/sdac_cross_task_summary_new.png')
    print('\nCross-task summary saved.')

    # Timing table
    print('\n── Training time (minutes) ─────────────────────────────────────')
    header = f"{'Variant':<22}" + ''.join(f"{n:<18}" for n, _ in _available)
    print(header)
    print('─' * len(header))
    for v in variants:
        row = f'{v:<22}'
        for _, hist in _available:
            if v in hist:
                row += f"{hist[v]['total_wall_time']/60:<18.1f}"
            else:
                row += f"{'N/A':<18}"
        print(row)